# Lumen Clip — Google Colab GPU backend
Temporary free GPU. Runtime → Change runtime type → GPU, then Runtime → Run all.
Keep this tab open while the website generates videos.

## 1. Install dependencies

In [ ]:
import subprocess, sys
pkgs=['diffusers>=0.29.0','transformers>=4.41.0','accelerate>=0.31.0','safetensors>=0.4.3','fastapi>=0.111.0','uvicorn>=0.30.0','imageio>=2.34.0','imageio-ffmpeg>=0.5.1','opencv-python-headless>=4.10.0','pydantic>=2.7.0']
subprocess.check_call([sys.executable,'-m','pip','install','-q',*pkgs])
print('dependencies ready')

## 2. GPU check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('PyTorch:', torch.__version__)
print('CUDA version:', torch.version.cuda)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2), 'GB')
else:
    raise SystemExit('No GPU. Runtime → Change runtime type → GPU, then Restart session.')

## 3. Download latest GitHub backend

In [ ]:
from pathlib import Path
import urllib.request, sys
REPO='https://raw.githubusercontent.com/sgue19000/t2v-kaggle-webapp/main/colab'
workdir=Path('/content')
for name in ('generator.py','server.py'):
    urllib.request.urlretrieve(f'{REPO}/{name}', workdir/name)
    print('updated', name)
if str(workdir) not in sys.path:
    sys.path.insert(0, str(workdir))

## 4. Load the video model

In [ ]:
from generator import load_pipeline, model_info, gpu_report
print(gpu_report())
load_pipeline()
print(model_info())

## 5. Tiny generation test

In [ ]:
from pathlib import Path
from IPython.display import Video, display
from generator import generate_video_with_fallback
demo=Path('/content/outputs/demo.mp4')
path, meta = generate_video_with_fallback({'prompt':'A golden retriever running through tall grass at sunrise','num_frames':8,'height':256,'width':256,'fps':8,'steps':12,'guidance_scale':9,'seed':42}, out_path=demo)
print(meta)
print('wrote', path, 'bytes', path.stat().st_size)
assert path.exists() and path.stat().st_size>1024
display(Video(str(path), embed=True))

## 6. Start FastAPI

In [ ]:
import os, time, threading
from pathlib import Path
os.environ['T2V_OUTPUT_DIR']='/content/outputs'
os.environ['T2V_SKIP_PRELOAD']='1'
Path('/content/outputs').mkdir(parents=True, exist_ok=True)
def run_api():
    import uvicorn
    from server import app
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')
threading.Thread(target=run_api, daemon=True).start()
time.sleep(3)
print('FastAPI listening on 0.0.0.0:8000')

## 7. Cloudflare Quick Tunnel

In [ ]:
import time, subprocess, re
from pathlib import Path
cf=Path('/content/cloudflared')
if not cf.exists():
    subprocess.check_call(['wget','-q','-O',str(cf),'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
    cf.chmod(0o755)
log_path=Path('/content/tunnel.log')
log=open(log_path,'w')
subprocess.Popen([str(cf),'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=log, stderr=subprocess.STDOUT)
url=None
for _ in range(45):
    time.sleep(1)
    found=re.findall(r'https://[-a-z0-9.]+trycloudflare.com', log_path.read_text(errors='ignore'))
    if found:
        url=found[-1]; break
if not url:
    raise SystemExit('Tunnel URL missing. Re-run this cell.')
print('='*40)
print('PUBLIC API URL')
print(url)
print('='*40)
globals()['PUBLIC_API_URL']=url

## 8. Test /health

In [ ]:
import json, urllib.request
data=json.load(urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=30))
print(json.dumps(data, indent=2))
assert data.get('ok') is True

## 9. Test /generate

In [ ]:
import json, time, urllib.request
req=urllib.request.Request('http://127.0.0.1:8000/generate', data=json.dumps({'prompt':'A cinematic futuristic city at night, flying cars, rain','num_frames':8,'width':256,'height':256,'fps':8,'steps':12,'guidance_scale':9,'seed':12345}).encode(), headers={'Content-Type':'application/json'}, method='POST')
started=json.load(urllib.request.urlopen(req, timeout=30))
print(started)
job_id=started['job_id']
for _ in range(120):
    snap=json.load(urllib.request.urlopen(f'http://127.0.0.1:8000/status/{job_id}', timeout=30))
    print(snap.get('status'), snap.get('progress'), snap.get('message') or snap.get('error'))
    if snap.get('status') in ('completed','failed'): break
    time.sleep(5)
print(snap)
globals()['LAST_JOB_ID']=job_id

## 10. Display generated MP4

In [ ]:
from IPython.display import Video, display
from pathlib import Path
job_id=globals().get('LAST_JOB_ID')
paths=[Path(f'/content/outputs/{job_id}.mp4')] if job_id else []
paths.append(Path('/content/outputs/demo.mp4'))
shown=False
for p in paths:
    if p.exists() and p.stat().st_size>0:
        print(p, p.stat().st_size); display(Video(str(p), embed=True)); shown=True; break
if not shown: print('No MP4 found')